# 📱 Teen Phone Addiction and Lifestyle Survey Analysis
A clean Google Colab notebook to:
- Load the dataset
- Inspect head and tail
- Check for missing values
- Understand schema design ideas


In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import joblib

### Data Processing

## Revisit data processing

### Subtask:
Revert the label encoding of 'Addiction_Level' and keep it as a numerical variable.


**Reasoning**:
Drop the label-encoded 'Addiction_Level' column from the current DataFrame and reload the original dataset to get the numerical 'Addiction_Level'.



In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/jkeza1/Group5_database-prediction/refs/heads/main/data/teen_phone_addiction_dataset.csv')

**Reasoning**:
Reapply the data processing steps to the reloaded DataFrame, excluding the label encoding of 'Addiction_Level'. This includes dropping columns, ordinal encoding, label encoding other categorical features, handling missing values, scaling, and feature engineering.



In [42]:
df.head()

,ID,Name,Age,Gender,Location,School_Grade,Daily_Usage_Hours,Sleep_Hours,Academic_Performance,Social_Interactions,...,Screen_Time_Before_Bed,Phone_Checks_Per_Day,Apps_Used_Daily,Time_on_Social_Media,Time_on_Gaming,Time_on_Education,Phone_Usage_Purpose,Family_Communication,Weekend_Usage_Hours,Addiction_Level
0,1,Shannon Francis,13,Female,Hansonfort,9th,4.0,6.1,78,5,...,1.4,86,19,3.6,1.7,1.2,Browsing,4,8.7,10.0
1,2,Scott Rodriguez,17,Female,Theodorefort,7th,5.5,6.5,70,5,...,0.9,96,9,1.1,4.0,1.8,Browsing,2,5.3,10.0
2,3,Adrian Knox,13,Other,Lindseystad,11th,5.8,5.5,93,8,...,0.5,137,8,0.3,1.5,0.4,Education,6,5.7,9.2
3,4,Brittany Hamilton,18,Female,West Anthony,12th,3.1,3.9,78,8,...,1.4,128,7,3.1,1.6,0.8,Social Media,8,3.0,9.8
4,5,Steven Smith,14,Other,Port Lindsaystad,9th,2.5,6.7,56,4,...,1.0,96,20,2.6,0.9,1.1,Gaming,10,3.7,8.6


In [43]:
df.tail()

,ID,Name,Age,Gender,Location,School_Grade,Daily_Usage_Hours,Sleep_Hours,Academic_Performance,Social_Interactions,...,Screen_Time_Before_Bed,Phone_Checks_Per_Day,Apps_Used_Daily,Time_on_Social_Media,Time_on_Gaming,Time_on_Education,Phone_Usage_Purpose,Family_Communication,Weekend_Usage_Hours,Addiction_Level
2995,2996,Jesus Yates,16,Female,New Jennifer,12th,3.9,6.4,53,4,...,0.3,80,15,2.7,1.8,1.0,Other,8,9.4,9.8
2996,2997,Bethany Murray,13,Female,Richardport,8th,3.6,7.3,93,5,...,0.9,45,8,3.1,0.0,0.3,Gaming,9,5.2,5.5
2997,2998,Norman Hughes,14,Other,Rebeccaton,7th,3.2,6.5,98,1,...,0.2,51,13,2.4,0.2,2.4,Social Media,9,5.9,6.2
2998,2999,Barbara Hinton,17,Female,Ramirezmouth,9th,6.7,7.5,67,3,...,1.6,125,17,1.7,2.6,1.5,Browsing,4,6.1,10.0
2999,3000,Curtis Johnson,17,Male,Lake Alexander,10th,3.5,6.9,79,4,...,0.6,117,8,0.0,2.3,0.1,Education,7,5.1,6.3


In [44]:
print(df.isnull().sum())

ID                        0
Name                      0
Age                       0
Gender                    0
Location                  0
School_Grade              0
Daily_Usage_Hours         0
Sleep_Hours               0
Academic_Performance      0
Social_Interactions       0
Exercise_Hours            0
Anxiety_Level             0
Depression_Level          0
Self_Esteem               0
Parental_Control          0
Screen_Time_Before_Bed    0
Phone_Checks_Per_Day      0
Apps_Used_Daily           0
Time_on_Social_Media      0
Time_on_Gaming            0
Time_on_Education         0
Phone_Usage_Purpose       0
Family_Communication      0
Weekend_Usage_Hours       0
Addiction_Level           0
dtype: int64


In [45]:
# 1. Drop irrelevant columns
df.drop(columns=['Name', 'Location'], inplace=True)

# 2. Encode categorical variables
# Ordinal encoding for School_Grade (7th–12th)
grade_order = ['7th', '8th', '9th', '10th', '11th', '12th']
df['School_Grade'] = OrdinalEncoder(categories=[grade_order]).fit_transform(df[['School_Grade']])

# Label encoding for Gender and Family_Communication
label_cols = ['Gender', 'Family_Communication']
for col in label_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# One-hot encode Phone_Usage_Purpose
df = pd.get_dummies(df, columns=["Phone_Usage_Purpose"], drop_first=True)

# 3. Handle missing values
df.fillna(df.mean(numeric_only=True), inplace=True)  # for numeric columns
for col in label_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# 4. Scale numerical features
# Exclude 'Addiction_Level' from scaling for now to verify its type later
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols.remove('Addiction_Level')
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# 5. Feature Engineering
df['Is_Heavy_User'] = (df['Daily_Usage_Hours'] > 5).astype(int)

/tmp/ipython-input-45-2509067882.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
/tmp/ipython-input-45-2509067882.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)',

**Reasoning**:
Verify that 'Addiction_Level' is a numerical column and the specified categorical features are one-hot encoded.



In [46]:
print(df['Addiction_Level'].dtype)
print(df.columns)

float64
Index(['ID', 'Age', 'Gender', 'School_Grade', 'Daily_Usage_Hours',
       'Sleep_Hours', 'Academic_Performance', 'Social_Interactions',
       'Exercise_Hours', 'Anxiety_Level', 'Depression_Level', 'Self_Esteem',
       'Parental_Control', 'Screen_Time_Before_Bed', 'Phone_Checks_Per_Day',
       'Apps_Used_Daily', 'Time_on_Social_Media', 'Time_on_Gaming',
       'Time_on_Education', 'Family_Communication', 'Weekend_Usage_Hours',
       'Addiction_Level', 'Phone_Usage_Purpose_Education',
       'Phone_Usage_Purpose_Gaming', 'Phone_Usage_Purpose_Other',
       'Phone_Usage_Purpose_Social Media', 'Is_Heavy_User'],
      dtype='object')


## Model selection

### Subtask:
Choose a suitable regression model (e.g., RandomForestRegressor).


**Reasoning**:
Import the necessary regression model and instantiate it.



In [47]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

## Train the model

### Subtask:
Train the chosen regression model on the prepared data.


**Reasoning**:
Fit the RandomForestRegressor model to the training data.



In [48]:
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

## Evaluate the model

### Subtask:
Evaluate the regression model using appropriate metrics (e.g., Mean Squared Error, R-squared).


**Reasoning**:
Import necessary metrics and calculate and print the MSE and R-squared for the regression model.



In [49]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse}")
print(f"R-squared (R2): {r2}")

Mean Squared Error (MSE): 34.648444500000004
R-squared (R2): 0.861895772331747


## Save the model

### Subtask:
Save the trained regression model.


**Reasoning**:
Save the trained regression model using joblib.



In [50]:
joblib.dump(model, "addiction_model_regression.pkl")

['addiction_model_regression.pkl']

## Summary:

### Data Analysis Key Findings

*   The 'Addiction_Level' column was successfully reverted to its numerical format (`float64`) after initially being label encoded.
*   A `RandomForestRegressor` model was chosen and successfully trained on the prepared data.
*   The model achieved a Mean Squared Error (MSE) of 34.648 and an R-squared ($R^2$) score of 0.862 on the test set.
*   The trained regression model was successfully saved as 'addiction\_model\_regression.pkl'.

### Insights or Next Steps

*   The R-squared score of 0.862 indicates that the model explains a significant portion of the variance in 'Addiction\_Level'. Further analysis could involve hyperparameter tuning to potentially improve performance.
*   Investigate feature importance from the trained `RandomForestRegressor` to understand which factors contribute most to predicting 'Addiction\_Level'.
